<a href="https://colab.research.google.com/github/sbhaidasna/CustomerServiceRefundAgent/blob/main/CustomerServiceRefundAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AI Refund Agent: End-to-End Notebook Documentation  


<a name="goal"></a>
# 🎯 **Goal**

The goal of this project was to take a problem that looks simple on the surface *'Can a customer get a refund?'* and use it as a vehicle to understand **the real mechanics of modern Agentic AI systems**. Having explored many low-code and no-code prototyping tools over the past few months, the objective here was to move beyond the interface layer and demonstrate a nuts-and-bolts, full-stack understanding of enterprise-grade AI architecture.

**Architectural Focus & Key Components:**
Built from the ground up with AI-assisted coding to maximize focus on systems principles over boilerplate code, this agent addresses critical production challenges by leveraging the following components:

- a **deterministic rules engine** that always gets the policy right  
- a **RAG layer** grounded in actual documentation, powered by  
  **NVIDIA Nemotron embeddings**, **FAISS**, and **Pinecone**  
- an **LLM explanation layer** that turns decisions into clear customer messaging  
- **observability** using **LangSmith** so every step can be traced, debugged, and measured  
- **evaluation harnesses** to validate both logic and LLM reasoning  
- and finally, an **MCP tool interface** so any agent or model can call this workflow as a service.

The deeper goal was not just to “build an agent,” but to answer questions every AI PM must grapple with:

- What should be **deterministic** vs **LLM-generated**?  
- How do grounding, retrieval, embeddings, and vector stores actually work together?  
- How do you measure latency, trace failures, and debug multi-step reasoning?  
- How do you make an AI system **explainable, reliable, and production-ready**?

By the end, this refund agent became a practical sandbox for learning the full stack of modern Agentic AI —>  from policy rules → RAG → LLMs → evaluations → vector DB swaps → observability → MCP interoperability, and a blueprint for how to design AI systems that are not only intelligent, but **transparent, testable, and trustworthy**.

Note: You'll need the API keys for **NVIDIA Nemotron, Pinecone, and LangSmith**. These services all offer free access tiers for development.

___
#**High-level Design**


<details>
<summary><strong>Click to expand - Design</strong></summary>

```
===============================================
            High-Level System Design
===============================================


        ┌──────────────────────────────────┐
        │    MCP Tool Interface (Stage 6)  │
        │  • refund_agent_api()            │
        │  • Standardized tool contract    │
        └──────────────┬───────────────────┘
                       │
                       ▼
        ┌──────────────────────────────────┐
        │   Agent Orchestrator (Stage 3)   │
        │  - Calls deterministic engine     │
        │  - Calls RAG + LLM explainer      │
        │  - Aggregates final response      │
        └──────────────┬───────────────────┘
                       │
      ┌────────────────┴────────────────────┐
      │                                     │
      ▼                                     ▼

┌───────────────────────────────┐   ┌─────────────────────────────────────┐
│ Deterministic Rules Engine     │   │        RAG Policy Retrieval         │
│ (Stage 3 Business Logic)       │   │     (Stages 2 + 5 Vector DBs)       │
│                               │   │                                     │
│ - lookup_order()              │   │ - Chunked policy documents          │
│ - check_refund_eligibility()  │   │ - NVIDIA embeddings                  │
│ - refund_decision_engine()    │   │                                     │
│ - issue_refund()              │   │ Vector DB Backends:                 │
│                               │   │   • FAISS (local, fast)             │
│ Output: structured decision   │   │   • Pinecone (cloud, persistent)    │
└─────────────────┬─────────────┘   │   active_retriever abstraction      │
                  │                 └───────────────────┬──────────────────┘
                  │                                     │ retrieved policy
                  ▼                                     ▼
        ┌──────────────────────────────────┐
        │    LLM Explanation Layer         │
        │     (Stage 1 + Stage 3 RAG)      │
        │ - Nemotron / Deepseek model      │
        │ - Grounded explanations           │
        │ - Combines:                      │
        │       • decision facts           │
        │       • retrieved policy chunks  │
        └──────────────┬──────────────────┘
                       │
                       ▼
         ┌─────────────────────────────────┐
         │   Final Customer-Ready Output   │
         │  (Decision + Policy Explanation)│
         └─────────────────────────────────┘


===============================================
       Observability & Performance Layer
===============================================

• Local latency profiling (Stage 4)
    - deterministic time
    - RAG time
    - LLM time

• LangSmith tracing (Stage 4)
    - refund-agent-end-to-end
    - FAISS vs Pinecone trace comparisons
    - Full pipeline visibility

===============================================
```
</details>


# 📑 Table of Contents  
(Click to jump to any section)

1. [Stage 1 — LLM Setup](#stage1)  
2. [Stage 2 — RAG Pipeline](#stage2)  
3. [Stage 3 — Refund Agent Workflow](#stage3)   
4. [Stage 4 — Observability, Performance, and LangSmith Tracing](#stage4)  
5. [Stage 5 — Swappable Vector Database Architecture (FAISS ↔ Pinecone)](#stage5)
6. [Stage 6 — Exposing the Refund Agent as an MCP Tool (Interoperable AI Service)](#stage6)


---
<a name="stage1"></a>
#**Stage 1: Initialize and Validate the LLM Stack**

Stage 1 sets up all the foundational components needed for working with NVIDIA Nemotron models through LangChain.
This includes:

* securely loading API keys
* configuring LangSmith tracing
* installing LLM connectors
* and validating the entire stack with a real inference call

This ensures the rest of the notebook runs on a fully working LLM backbone.

---

## **1. Environment & Credential Setup**

We configure:

* **NVIDIA_API_KEY** → Used to authenticate and call Nemotron models
* **LANGCHAIN_TRACING_V2** and **LangSmith keys** → Enable observability and LLM call tracing
* **LANGCHAIN_PROJECT** → Groups all traces under a single project in LangSmith

I used Colab Secrets to save the keys and later integrated this project to Github.

---

## **2. Install LangChain and NVIDIA AI Endpoints**

We install two essential libraries:

* `langchain` → LLM orchestration framework
* `langchain-nvidia-ai-endpoints` → Connector for NVIDIA’s hosted inference APIs

These allow the notebook to send prompts and receive model responses using a unified interface.

---

## **3. Test NVIDIA Nemotron Connectivity**

We instantiate a Nemotron LLM:

```python
llm = ChatNVIDIA(model="deepseek-ai/deepseek-v3.1")
```

Then perform a test inference to verify that:

* API keys are correct
* the model is reachable
* LangChain integration works

This is the “sanity check” before building RAG or business logic layers.

---

<a name="stage2"></a>

#**Stage 2: Build a RAG System Using FAISS + NVIDIA Embeddings**

Stage 2 transforms a static refund policy document into a fully queryable knowledge base using RAG (Retrieval-Augmented Generation).

We lay the foundation for all future agent behavior by:

* embedding the policy
* storing it in a vector database
* enabling semantic search
* validating retrieval and grounded LLM answers

---

## **2.1 Install and Validate FAISS (Local Vector Database)**

FAISS serves as the **in-memory semantic search engine**.
It enables fast retrieval of relevant policy chunks based on embeddings.

We:

* install `faiss-cpu`
* verify imports
* import LangChain’s `FAISS` wrapper

Later, in Stage 5, we make the vector store swappable with Pinecone (cloud database - persistent and scalable).

---

## **2.2 Define the Official Refund Policy Document**

We create `refund_policy_text`, a single source of truth containing:

* global rules
* region-specific rules
* additional operational rules

This is the document the agent must use exclusively when answering policy questions.

> In production, this would live in your internal knoweldgebase.
> Here, it is defined as a simple multi-line string.

---

## **2.3 Chunk the Policy for Retrieval**

A document too large cannot be embedded or retrieved efficiently.
We therefore:

* install `langchain-text-splitters`
* use `RecursiveCharacterTextSplitter` with

  * `chunk_size = 400`
  * `chunk_overlap = 80`

Chunking achieves:

* better semantic search
* localized context
* higher retrieval accuracy

We verify:

* number of chunks
* the first chunk’s content

---

## **2.4 Generate Embeddings (NVIDIA Embedding Model)**

We instantiate:

```
nvidia/nv-embedqa-e5-v5
```

This model converts text into high-dimensional vectors.
These vectors capture semantic meaning and are stored in the vector database. You can choose any other relevant Embedding model from the NVIDIA catalog.

We validate:

* embedding shape
* whether the model successfully embeds test strings

---

## **2.5 Build FAISS Vector Store & Retriever**

We:

* pass all chunks + embeddings into FAISS
* configure a retriever returning top-3 similar chunks

This enables semantic lookup when the user asks a question.

The retriever is also assigned to:

```
active_retriever = policy_retriever
```

Later (Stage 5), this becomes the switch that lets us toggle between FAISS(in-mmeory) and Pinecone(managed cloud-hosted Vector DB).

---

## **2.6 Retrieval Quality Test**

We define `preview_retrieval()` to inspect:

* which chunks are retrieved
* whether they make semantic sense
* whether chunking worked correctly

A simple test question like `"Print chunks"` confirms that retrieval is functioning.

---

## **2.7 Implement the Core RAG Function**

`refund_policy_rag_answer(question)` orchestrates:

1. **Retrieve** the top chunks via `active_retriever`
2. **Build a grounded prompt**
3. **Run the LLM** with strict grounding instructions

This ensures:

* answers come **only** from policy text
* no hallucinations
* agent follows operational rules (ask follow-up, route to human if needed)

This RAG function becomes the backbone for policy-aware customer responses.

---

## **2.8 Grounded Q&A Validation**

We run several sample questions to confirm:

* Retrieval returns the right context
* LLM respects grounding instructions
* Policy inconsistencies lead to safe fallback behavior
* The system handles no-answer cases cleanly

This completes the standalone RAG pipeline.

---
<a name="stage3"></a>

#**Stage 3 — Build the Deterministic Refund Decision Engine + Policy-Grounded Explanations**


Stage 3 transforms the RAG system from Stage 2 into a **full, end-to-end refund-processing agent**.
This introduces:

* a synthetic “orders database”
* deterministic rule enforcement
* refund execution & audit logging
* customer-friendly LLM explanations grounded in real policy
* formal evaluations of correctness

This is where the system becomes a *true agent* rather than a simple policy Q&A bot.

---

# **3.1 Create a Synthetic Orders Database (Transactional Grounding Layer)**

We simulate a real transactional backend using an **in-memory list of order dictionaries**.
Each order includes:

* `order_id`
* `customer_email`
* `country`
* `delivery_date`
* `is_defective`
* `already_refunded`
* `amount`

This provides **concrete facts** needed for refund eligibility decisions.

Using `delivery_date` (instead of shipment date) ensures all calculations are based on **policy-accurate time windows**.

This stage enables the agent to answer real-world customer questions such as:

> “I received my product on Nov 10 — am I still eligible for a refund?”

The synthetic DB also allows repeatable testing before any integration with real systems.

---

# **3.2 Implement Deterministic Business Logic**

This is the **core engine** of the refund agent.

###3.2.1 `lookup_order(order_id)`

Simulates a database/API lookup.

* Returns the matching order dict
* Returns `None` if the order does not exist

This isolates order access from refund logic, maintaining a clear abstraction boundary.

---

###3.2.2 `check_refund_eligibility(order, today)`

Implements the **entire refund policy in pure code**, ensuring:

* determinism
* auditability
* correctness
* reproducibility

It enforces:

* 30-day US standard window
* 45-day US defective window
* 60-day EU window
* delivered-only refunds
* “already refunded” = auto rejection
* time-window calculations using **delivery_date**

This function is the **source of truth** for causal refund eligibility decisions.

---

###3.2.3 `issue_refund(order, amount)`

Simulates the operational system action:

* marks the order as `already_refunded = True`
* appends a structured event to `refund_log`
* returns a refund record

This allows testing of:

* idempotency
* state changes
* refund transaction flows

---

###3.2.4 `refund_decision_engine(order_id, claimed_defective)`

This orchestrates the full deterministic workflow:

1. **Order lookup**
2. **Apply claimed defect flag**
3. **Eligibility evaluation**
4. **Issue refund (if eligible)**
5. **Return structured decision as a JSON-like dictionary**

Outputs include:

* `status` (approved/rejected)
* `reason`
* `refund_amount`
* `days_since_delivery`
* `window_days`
* `refunded` (boolean)

This is the *non-LLM action core* of the agent — guaranteed free of hallucinations.

---

#**3.3 Combine Deterministic Decisions + RAG + LLM Explanations**


The customer sees **an explanation**, not raw JSON.
So we layer LLM reasoning *on top of deterministic logic* to produce grounded, transparent messaging.

###Function: `explain_refund_decision_with_policy(order_id, claimed_defective)`

This function:

1. Calls the deterministic engine → gets structured facts
2. Retrieves relevant policy chunks via RAG
3. Builds a hybrid prompt containing:

   * official policy text
   * decision facts (days, windows, reasons, amounts)
4. Uses the LLM to produce a:

   * clear
   * empathetic
   * policy-grounded
   * customer-ready explanation

Examples:

* “Your refund was approved…”
* “Your refund was declined because your order is outside the 30-day window…”

This architecture is **ideal for enterprise AI**:

* LLM = language + clarity
* Code = decisions + compliance

LLMs *do not* decide refunds — they only explain them.



---

#**3.4 Smoke Test — Full Pipeline (Decision Engine + RAG + LLM)**



We now simulate real customer flows:

* valid US refunds
* defective vs non-defective
* EU refund rules
* already-refunded orders
* unknown orders

Each test confirms:

* deterministic logic returns the correct decision
* RAG retrieves the right policy chunks
* LLM explains the decision accurately

This validates the entire stack *before* introducing advanced evaluation or instrumentation.

</details>

---

# **3.5 Deterministic Evaluation Suite**


To ensure reliability, we create a small evaluation framework that tests:

* correct approvals
* correct rejections
* edge cases (unknown orders, already refunded orders)
* eligibility window boundaries

### Why deterministic evals matter:

* catch logic regressions
* verify correctness after changes
* ensure the agent is **trustworthy**
* guarantee consistency across scenarios

Each test case specifies:

* `order_id`
* `claimed_defective`
* `expected_status`

The eval runner prints:

* Expected vs actual
* PASS/FAIL per case
* Summary totals

By adding **approved cases** (e.g., EU inside window), we ensure we test *both* branches.

---

# **Summary of Stage 3


By the end of Stage 3, the agent becomes a complete, end-to-end refund decision system with:

### Deterministic foundations

* synthetic orders DB
* hard-coded policy engine
* stateful refund logging

### AI-augmented explanation layer

* RAG to fetch relevant policy rules
* LLM to generate customer-friendly messaging
* no hallucinations in core decisions

### Evaluation + Observability scaffolding

* deterministic evals
* scenario coverage
* state mutation (refunds) in a controlled environment

Stage 3 sets the foundation for:

* Stage 4: LangSmith observability
* Stage 5: FAISS ↔ Pinecone pluggable backends
* Stage 6: MCP tool interface

All later stages build on Stage 3’s clean deterministic core.


---
<a name="stage4"></a>


# 🧩 **Stage 4 — Observability, Latency Profiling & LangSmith Tracing**


Stage 4 enhances our refund agent with **observability**, **performance measurement**, and **full tracing** using LangSmith.
This turns our working agent into a **debuggable, measurable, production-aware AI system**.

We introduce observability **without modifying core business logic** — a critical PM + engineering principle.

---

# **4.1 Local Timing Instrumentation (Latency Profiling)**

Before introducing external tooling, we add *local instrumentation* to measure:

* **Deterministic logic latency**
  (`refund_decision_engine`, order lookup, policy rules)

* **RAG + LLM latency**
  (retrieval + explanation generation)

* **Total end-to-end latency**

This is handled by the function:

### `timed_explain_refund(order_id, claimed_defective)`

It returns both:

* the agent’s decision + explanation
* a timing breakdown of each component

### Why this matters

This lets us answer PM-level performance questions:

* Where is most of the latency?
* How much overhead does RAG add?
* Is the LLM the bottleneck?
* How does performance change by scenario?

(This is essential for PMs doing *cost modeling, user experience planning, and SLA analysis*.)

We then run **multi-scenario latency tests** across:

* US inside/outside window
* EU inside/outside window
* defective vs non-defective
* already-refunded orders
* unknown orders

This creates our first performance baseline.

---

# **4.2 Clean Separation: Logic vs Instrumentation**

A key architectural design principle:

> **Instrumentation must not pollute business logic.**

This is why:

* All timing code is **separate**
* The refund engine remains pure + deterministic
* Profiling is optional and only invoked explicitly
* The code stays production-safe

This mirrors real-world microservice & agent design:
“Observability wraps the system; it never *lives inside* the system.”

---

# **4.3 LangSmith Project Setup and Connectivity**

Before enabling tracing, we verify that Colab can communicate with LangSmith:

* Load API keys from environment variables
* Initialize the LangSmith client
* Ensure the project (`refund-agent-demo`) exists
* Write a *manual test run* to confirm connectivity

This step guarantees LangSmith is ready to ingest traces.

---

# **4.4 End-to-End Tracing with `@traceable`**

We wrap the agent entrypoint:

### `run_refund_agent(order_id, claimed_defective)`

with:

```python
@traceable(run_type="chain", project_name="refund-agent-demo")
```

Now LangSmith automatically logs:

* the RAG retrieval
* LLM inputs and outputs
* timing for each step
* intermediate data flow
* errors and exceptions

### Benefits:

* You can inspect **what the LLM saw**
* You can compare **retrieved chunks vs final explanation**
* You can view **latency per component**
* You gain visibility into **multi-step agent reasoning**

This turns your notebook into a **fully instrumented AI system** with:

* debugging
* analytics
* evaluation readiness
* auditability

Just like a production agent.

---

# **4.5 Why Observability Matters (PM Perspective)**

Modern AI systems behave like pipelines, not single model calls.

Observability provides answers to critical PM questions:

### Product correctness

* Why did the agent approve/reject a refund?
* Did it use the correct policy chunk?
* Was the explanation aligned with rules?

### Performance & UX

* What slows down the agent?
* How does latency vary across scenarios?
* What is the user-perceived wait time?

### Debuggability

* What intermediate steps occurred?
* Did retrieval return the correct context?
* Did the LLM hallucinate or contradict policy?

### Stability & Monitoring

* Are failures predictable?
* How often does the agent route to human support?
* Are there regressions after policy updates?

Stage 4 provides the **tooling and visibility** needed to answer all of these.

---

# ✔️ Summary of Stage 4

At the end of Stage 4, the refund agent is now:

### 🔍 **Observable**

Full tracing of decision → RAG → LLM steps.

### ⏱ **Measurable**

Latency breakdown for deterministic vs RAG+LLM paths.

### 🧪 **Evaluable**

Ready for future automated LangSmith evaluations.

### 🛠 **Debuggable**

Clear visibility into inputs, outputs, and intermediate reasoning.

### 🧱 **Production-aware**

With separation of concerns between logic and instrumentation.

This sets the foundation for:

* **Stage 5** — Pluggable Vector Databases (FAISS ↔ Pinecone)
* **Stage 6** — Exposing the Agent as an MCP Tool


---

<a name="stage5"></a>

#**Stage 5 — Swappable Vector Database Architecture (FAISS ↔ Pinecone)**


Stage 5 introduces a major architectural upgrade:
the ability to **seamlessly switch** between:

* a **local, in-memory FAISS vector store** (fast, free, ideal for development)
* a **cloud-hosted Pinecone vector database** (scalable, persistent, production-grade)

This mirrors real-world enterprise RAG deployments, where PMs must understand how system behavior changes when moving from local prototyping to managed infrastructure.

---

# **5.1 Install Pinecone Client + LangChain Integration**

We install:

* `pinecone-client` → communicates with Pinecone’s managed service
* `langchain-pinecone` → gives us a vector store wrapper consistent with LangChain
* `grpc` extras → improves Pinecone performance

This ensures our notebook can:

* connect to Pinecone
* create vector stores
* insert embeddings
* search them at inference time

PM takeaway:

> You’ve now enabled your agent to run on a production-grade vector database.

---

# **5.2 Configure Pinecone Credentials + Index Name**

We set:

* `PINECONE_API_KEY`
* `PINECONE_INDEX_NAME`

Your Pinecone index must already be created in the Pinecone Console (free tier supports small indexes).

This step externalizes configuration, enabling clean environment-driven behavior:

> **No code changes needed to switch vector DB — only environment & flags.**

This is exactly how production services toggle between dev/staging/prod clusters.

---

# **5.3 Build a Pinecone Vector Store from Policy Documents**

We define:

### `build_pinecone_vectorstore(policy_docs, embeddings, index_name)`

What it does:

1. Connects to Pinecone using your API key
2. Reuses an existing index (the recommended production pattern)
3. Uploads all policy chunks + embeddings into a persistent namespace
4. Returns a **PineconeVectorStore**
5. Creates a **retriever** via `.as_retriever()`

This gives us a cloud-hosted version of our RAG knowledge base.

### Why this matters

* FAISS is ephemeral — disappears when Colab restarts
* Pinecone is persistent — survives across runs and deployments
* Pinecone scales to millions/billions of vectors
* Pinecone supports replication, filtering, hybrid search, etc.

This is the step where the notebook starts looking like a **real AI backend**.

---

# **5.4 Define a Swappable Retriever Layer (FAISS ↔ Pinecone)**

A simple but powerful design pattern:

```python
USE_PINECONE = True
active_retriever = pinecone_retriever if USE_PINECONE else faiss_retriever
```

Now the entire RAG pipeline switches automatically based on one flag.

Everything downstream — including:

* refund explanations
* evals
* LangSmith traces
* latency tests

…all work identically.

### PM takeaway

This is the same level of abstraction used in enterprise platforms to allow:

* hybrid deployments
* A/B testing of vector backends
* failover logic
* environment isolation

You now have *infrastructure-agnostic* RAG.

---

# **5.5 Compare FAISS vs Pinecone Retrieval Behavior**

We run the same questions against both vector backends:

* US refund window
* EU refund window
* defective product logic

Expected outcome:

* Answers should be identical or extremely close
* Retrieval consistency should validate correct indexing

This validates that your embedding → indexing → retrieval pipeline behaves the same regardless of backend.

### Why this matters

If a back-end change alters behavior, that is a **critical regression** in production systems.

We are now validating backend-comparability like a real AI infra engineer.

---

# **5.6 LangSmith-Instrumented A/B Testing: FAISS vs Pinecone**

This is the capstone of Stage 5.

I immplemented:

### `traced_refund_policy_rag_faiss(question)`

### `traced_refund_policy_rag_pinecone(question)`

Both decorated with:

```python
@traceable(project_name="refund-agent-demo")
```

This produces *separate traces* in LangSmith for:

* FAISS retrieval path
* Pinecone retrieval path

Each trace logs:

* retriever latency
* LLM call latency
* retrieved documents
* final answer

### Why this is significant

You can now **quantitatively compare** FAISS vs Pinecone:

| Property      | FAISS            | Pinecone              |
| ------------- | ---------------- | --------------------- |
| Runtime       | In-memory (fast) | Network call (slower) |
| Persistence   | None             | Yes                   |
| Scalability   | Limited          | Large-scale           |
| Cost          | Free             | Free/Paid tiers       |
| Observability | Local            | Cloud-native          |

This is the kind of infrastructure comparison a senior PM or AI architect would perform when evaluating RAG systems.

---

# **Summary of Stage 5**

By the end of Stage 5, the agent supports:

### 🔁 **Pluggable vector backends**

Switch FAISS ↔ Pinecone with a single flag.

### 🌩 **Managed vector DB (Pinecone)**

Persistent, scalable RAG ready for real deployment.

### 🧪 **Backend consistency testing**

Validate retrieval and output behavior across DBs.

### 📊 **LangSmith-instrumented A/B evaluation**

Compare latency and behavior of FAISS vs Pinecone with full traces.

### 🔨 **Production architecture patterns**

* environment-based config
* retriever abstraction
* cloud/local swap
* performance profiling

This stage elevates the notebook from a prototype into an **enterprise-quality AI architecture that supports scaling, swapping, and observability**.

</details>

---

<a name="stage6"></a>
# 🧩 **Stage 6 — Exposing the Refund Agent as an MCP Tool (Interoperable AI Service)**

Up to now, the refund agent lived entirely inside a Colab notebook.

But real AI ecosystems require **interoperability**:
agents must be callable by *other agents, tools, models, or orchestrators*.

This is where **MCP (Model Context Protocol)** comes in.

---

# **6.1 What We Are Doing in This Stage**

In Stage 6, we:

1. **Identify the minimal API surface** that an external system needs
2. **Wrap our entire refund workflow** into a clean, structured function
3. Prepare to expose this wrapper via **an MCP server** (in a `.py` file)

This means:

### Instead of “an LLM calling Python functions,”

we shift to:

### “Any agent / LLM / orchestrator calling a formal MCP tool.”

This is a *critical professional skill* for AI PMs and architects:
* exposing your AI logic as a reusable capability inside larger agent systems

The MCP tool must:

* be **deterministic**
* take **simple inputs**
* return **JSON-serializable outputs**
* expose **both the decision and the explanation**
* hide notebook implementation details

The entire refund agent collapses into one external-facing function:

```
refund_agent_api(order_id, claimed_defective) → structured result
```

This becomes *the contract* between this AI system and any other system.


This stage extracts the final, external-facing API:

* receives simple inputs
* runs deterministic policy logic
* runs RAG + LLM explanation
* merges everything into a **clean JSON dictionary**
* later: becomes callable via MCP protocol

This makes the agent ready for:

* integration
* automation
* orchestration
* productionization

---

# ✔️ Summary

Stage 6 is where the notebook becomes a **real AI service**:

* clean API
* deterministic core
* grounded LLM explanations
* MCP-compatible response schema

In Stage 6 Step 2, I will wrap this inside a full MCP server (`refund_agent_mcp_server.py`) - on Github




In [ ]:
# Stage 1 - Initialization and Validation of the LLM Stack.
#This section handles environment setup, securely configuring the necessary API credentials,
#and executing a preliminary API call to validate the connection with NVIDIA Nemotron/LLM

import os
from google.colab import userdata

# Set up NVIDIA API key to access Nemotron models
# Load the key from Google colab secrets.

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

# LangChain → LangSmith tracing (used by LangChain integrations)
# Will do instrumentation with LangSmith
# Load the key from Google colab secrets.

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "refund-agent-demo"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")

# LangSmith-native tracing (used by @traceable etc.)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.environ["LANGCHAIN_API_KEY"]


#Use the pip package manager to install necessary libraries:
# -q: Quiet output (less distracting messages).
# -U: Upgrade packages if they are already installed.
# - langchain: The core framework for developing applications powered by LLMs.
# - langchain-nvidia-ai-endpoints: The specific connector to interface with NVIDIA's hosted LLM services.
!pip install -qU \
  langchain \
  langchain-nvidia-ai-endpoints

# Setting up, invoking and validating the LLM call
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Starting off with Deepseek. You can check the NVIDIA catalog for the other available models.
MODEL_NAME = "deepseek-ai/deepseek-v3.1"  # example from NVIDIA's catalog

llm = ChatNVIDIA(model=MODEL_NAME)

#Let's test the call
response = llm.invoke("Howdy! Say Hello to the Product Manager building their AI agent from scratch!")
print(response)

In [ ]:
# Stage 2 - Step 1 -  Vector Database Setup and Validation (FAISS). This section initializes the connection logic for the local, in-memory FAISS vector store.
# We'll start with in-memory FAISS for the demo and then later also add cloud-hosted Pinecone managed vectorDB connection. We'll manage which option to leverage by setting an env variable
# This step installs the FAISS library (for efficient similarity search) and its LangChain community wrapper. A test is then performed to confirm that the vector store dependencies are correctly installed and ready for use in a RAG system

!pip install -qU faiss-cpu langchain-community

# Simple FAISS import test
try:
    import faiss
    print("FAISS imported successfully — you are good to go!")
    print("FAISS version:", faiss.__version__)
except ImportError as e:
    print("FAISS failed to import:", e)

from langchain_community.vectorstores import FAISS

print("LangChain FAISS wrapper imported successfully.")


In [ ]:
# Stage 2 - Step 2 - Document Preparation: Defining the Knowledge Source - setting up the refund policy for the product
# Although, here in-memory, think of this as a/set of document(s) that could reside on your internal sharepoint
# This step defines the context document (refund_policy_text) that the LLM will retrieve information from.
# This document represents a single source of truth (e.g., an internal SharePoint file or knowledge base) that will be indexed and stored in the vector database for RAG

refund_policy_text = """

Refund Policy v1

Global rules:
- Standard products: Customers can request a refund within 30 days from the delivery date.
- Defective products: Customers can request a refund within 45 days from the delivery date.

Region-specific rules:
- Europe (EU): Customers can request a refund within 60 days from the delivery date, whether or not the product is defective.
- United States (US): Follow the global rules (30 days standard, 45 days defective).
- Other regions: Default to global rules unless explicitly overridden.

Additional rules:
- Refunds can only be issued for orders that have been delivered.
- Already refunded orders can't be refunded again.
- If a refund is requested outside the allowed window, the agent must kindly decline the refund and clearly explain why. If the customer insists on refund despite that, please route them to the human agent.
- If the policy text is unclear or missing for a specific scenario, the agent should say it is unsure rather than making up a rule and route the query to the human agent.
"""

print(refund_policy_text)


In [ ]:
#Stage 2 - Step 3 - Chunking the knowledge source/policy
#This section prepares the raw policy document for storage in the vector database.
#We use the Recursive Character Text Splitter to break the long document into smaller, semantically coherent chunks (of 400 characters with an 80 character overlap).
#This process is essential for efficient vector indexing and for ensuring the LLM receives highly relevant context during retrieval.

!pip install -qU langchain-text-splitters

#Call Reursive Character Text Splitter for chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Imported RecursiveCharacterTextSplitter successfully!")

text_splitter = RecursiveCharacterTextSplitter(
    # Ideal chunk size for policy documents is often 500-1000 characters/tokens
    chunk_size=700,
    # Overlap is critical to prevent loss of context across boundaries
    chunk_overlap=150,
    # Prioritize breaking at paragraph/section breaks first
    separators=[
        "\n\n",  # Two newlines (paragraph break)
        "\n",    # Single newline (line break/list item break)
        " ",     # Space
        "",      # Character fallback
        ],
)

#Chunk
policy_docs = text_splitter.create_documents([refund_policy_text])

#Verify
print(f"Number of chunks: {len(policy_docs)}")
print("First chunk:\n")
print(policy_docs[0].page_content)

In [ ]:
#Stage 2 - Step 4 - Configuring and generating text embeddings. Generated embeddings using NVIDIA embedding models. You can use any relevant text embedding model available on Nemotron.
#This step instantiates the NVIDIA Embeddings model (nvidia/nv-embedqa-e5-v5).
#This component is responsible for vectorization —> converting the text chunks into high-dimensional numerical arrays (vectors/embeddings).
#These vectors capture the semantic meaning of the text and are what the vector store (FAISS/Pinecone) will use to perform similarity search during the retrieval process.

from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

EMBED_MODEL = "nvidia/nv-embedqa-e5-v5"

embeddings = NVIDIAEmbeddings(model=EMBED_MODEL)

print("Testing if the embedding model works")
try:
    test_vectors = embeddings.embed_documents(["hello world", "refund policy"])
    print("Embedding returned", len(test_vectors), "vectors.")
    print("Vector[0] length:", len(test_vectors[0]))
except Exception as e:
    print("Embedding failed:", e)

In [ ]:
#Stage 2 - Step 5 - Building the in-memory FAISS vector store
#The embedded policy chunks are loaded into the FAISS index, effectively creating the in-memory vector database.
#A retriever object is instantiated from this database, specifying that it will look up the top 3 closest matching documents (k=3) when receiving a query from the LLM chain

from langchain_community.vectorstores import FAISS

policy_vectorstore = FAISS.from_documents(policy_docs, embeddings)

policy_retriever = policy_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},  # return top 5 chunks
)

active_retriever = policy_retriever # Initialize active_retriever here

print("Policy vector store ready with", len(policy_docs), "chunks.")

In [ ]:
#Stage 2 - Step 6 - Retrieval test - Validate the chunk quality and similarity search.
#This function, preview_retrieval, is designed to validate the performance of the policy_retriever.
#It simulates a user query being sent to the vector store and prints the top k most relevant context chunks returned (as configured in the previous step).
# This test ensures the indexing and semantic search functionality are working correctly before connecting to the final LLM.

def preview_retrieval(query: str):
    print(f"USER QUESTION: {query}\n")

    # New LangChain: retrievers are Runnables → use .invoke()
    docs = policy_retriever.invoke(query)

    for i, d in enumerate(docs, start=1):
        print(f"--- Retrieved chunk {i} ---")
        print(d.page_content)
        print()

# Print chunks
preview_retrieval("Print chunks")


In [ ]:
#Stage 2 - Step 7 - Implement the RAG answer function
"""Implementing the Core Retrieval-Augmented Generation (RAG) Function
This function, refund_policy_rag_answer, defines the complete RAG execution pipeline. It orchestrates the flow:
Retrieval: The user's question is used by the active_retriever to fetch the most relevant policy context chunks.
Augmentation: The retrieved context is injected into a specific System Prompt along with strict instructions (grounding).
Generation: The complete, augmented prompt is sent to the LLM (llm.invoke) to generate an answer based ONLY on the provided context, ensuring accuracy and adherence to the source document.
"""

def refund_policy_rag_answer(question: str) -> str:
    # 1. Retrieve relevant policy chunks (new API: .invoke)
    #context_docs = policy_retriever.invoke(question)
    context_docs = active_retriever.invoke(question)
    context_text = "\n\n".join(d.page_content for d in context_docs)

    # 2. Build a grounded prompt
    prompt = f"""
You are a customer service assistant that must answer questions about refund policy.
You are given the official refund policy context below. This is the source of truth:
<refund_policy_context>
{context_text}
</refund_policy_context>

Instructions:
- If you are unclear or do not have enough information e.g. no region specified, please ask follow up questions.
- Answer the user's question using ONLY the refund policy context provided.
- If the context does not contain enough information to answer confidently, say:
  "I don't know based on the provided refund policy. Let me connect you with the live agent"
- Do NOT invent new rules or guess or embellish.

User question: {question}
"""

    # 3. Call the LLM to test the RAG function
    response = llm.invoke(prompt)
    return str(response)


In [ ]:
#Stage 2 - Step 8 - Run some example Q&As to check for groundedness
"""End-to-End Validation: Groundedness Testing
This loop executes a series of sample Q&A to perform a final end-to-end validation of the RAG system.
The primary goal is to verify groundedness—confirming the LLM's responses are derived only from the retrieved policy text and that the agent
correctly routes or declines queries when the context is insufficient (as defined in the system prompt).
"""

questions = [
    "My product was delivered on 1st Oct 2025 which is later than I wanted, my product is not defective but I don't want it anymore, please refund",
    "What is the refund window for a standard product in the US?",
    "What is the refund window for a defective product in the US?",
    "What is the refund window for any product in Europe?",
    "Can a refund be issued if the order was never shipped?",
    "What should the agent do if the policy text is unclear for a scenario?",

]

for q in questions:
    print("Q:", q)
    print("A:", refund_policy_rag_answer(q))
    print("=" * 80)


In [ ]:
#Stage 3 - Step 1 - Create a synthetic “orders database” here to start with instead of heavy DB scenario. Can be revised later.
"""This section defines a synthetic, in-memory list of dictionaries (orders) to act as the primary transactional database for the refund agent.
This mock data source is used to ground the refund decisions by providing real-time order status, delivery dates, and policy-relevant details (like country and defect status).
This approach allows for rapid development and testing before integrating with a heavy, external db.
"""

# Set system date
from datetime import date, timedelta
print("System date:", date.today())

from datetime import date

#Creating orders to demonstrate scenarios based on the system date today

orders = [
    {
        "order_id": "ORD-1001",
        "customer_email": "alice@example.com",
        "country": "US",
        "delivery_date": date(2025, 11, 10),  # may or may not be in 30d window depending on the run/system date
        "is_defective": False,
        "amount": 100.0,
        "already_refunded": False,
    },
    {
        "order_id": "ORD-1002",
        "customer_email": "bob@example.com",
        "country": "US",
        "delivery_date": date(2025, 10, 1),   # outside 45-day window
        "is_defective": True,
        "amount": 250.0,
        "already_refunded": False,
    },
    {
        "order_id": "ORD-1003",
        "customer_email": "carla@example.com",
        "country": "EU",
        "delivery_date": date(2025, 11, 5),   # inside 60-day EU window
        "is_defective": False,
        "amount": 80.0,
        "already_refunded": False,
    },
    {
        "order_id": "ORD-1004",
        "customer_email": "dan@example.com",
        "country": "EU",
        "delivery_date": date(2025, 8, 1),    # outside 60-day window
        "is_defective": True,
        "amount": 300.0,
        "already_refunded": False,
    },
    {
        "order_id": "ORD-1005",
        "customer_email": "erin@example.com",
        "country": "US",
        "delivery_date": date(2025, 11, 30),  # very recent
        "is_defective": True,
        "amount": 50.0,
        "already_refunded": True,  # already refunded once
    },
    {
        "order_id": "ORD-1006",
        "customer_email": "jb@example.com",
        "country": "US",
        "delivery_date": date(2025, 11, 30),  # very recent - US
        "is_defective": False,
        "amount": 58.0,
        "already_refunded": False,
    },
    {
        "order_id": "ORD-1007",
        "customer_email": "sb@example.com",
        "country": "EU",
        "delivery_date": date(2025, 11, 1),  # very recent - EU
        "is_defective": False,
        "amount": 200.0,
        "already_refunded": False,
    }


]

print(f"Loaded {len(orders)} synthetic orders.")
orders[:2]


In [ ]:
#Stage 3 - Step 2.1 - a simple function lookup_order helper that agent can use to check if an order is real and fetch its details. This is the stand-in for a real database/API call.
"""
Utility Function for Order Lookups
Defines the lookup_order helper function, which simulates an API call to a database to fetch specific, real-time details of an order, returning the order dictionary or None if the ID is invalid.
"""

def lookup_order(order_id: str):
    """
    Look up an order by ID in our synthetic orders list.
    Returns the order dict if found, otherwise None.
    """
    for order in orders:
        if order["order_id"] == order_id:
            return order
    return None

# Quick tests
print("ORD-1001 ->", lookup_order("ORD-1001"))
print("ORD-9999 ->", lookup_order("ORD-9999"))


In [ ]:
#Stage 3 - Step 2.2 - check_refund_eligibility (uses system date by default)
"""This function, check_refund_eligibility, serves as the hardcoded business logic layer that applies the specific policy rules (from Stage 2) against the facts retrieved from the mock database (from Stage 3).
It performs the time-based calculations using the delivery_date and applies conditional logic (based on country and defect status) to accurately determine refund eligibility.
This explicit, non-LLM logic ensures the agent's decisions are deterministic always, verifiable and policy-compliant.
"""

from datetime import date

def check_refund_eligibility(order: dict, today: date = None) -> dict:
    """
    Apply refund policy rules to decide if a given order is eligible for a refund.
    Uses today's date by default unless a specific date is provided.
    """
    if today is None:
        today = date.today()

    if order is None:
        return {
            "eligible": False,
            "reason": "Order not found.",
            "window_days": None,
            "days_since_delivery": None,
        }

    if order["already_refunded"]:
        return {
            "eligible": False,
            "reason": "Order has already been refunded.",
            "window_days": None,
            "days_since_delivery": None,
        }

    delivery_date = order["delivery_date"]
    if delivery_date is None:
        return {
            "eligible": False,
            "reason": "Order has not been delivered yet; refunds can only be issued for delivered orders.",
            "window_days": None,
            "days_since_delivery": None,
        }

    days_since_delivery = (today - delivery_date).days
    country = order["country"]
    is_defective = order["is_defective"]

    # Determine applicable window from your policy:
    # - EU: 60 days for any product
    # - Global (e.g., US): 30 days standard, 45 days defective
    if country == "EU":
        window_days = 60
    else:
        window_days = 45 if is_defective else 30

    if days_since_delivery <= window_days:
        return {
            "eligible": True,
            "reason": f"Order is within the {window_days}-day refund window.",
            "window_days": window_days,
            "days_since_delivery": days_since_delivery,
        }
    else:
        return {
            "eligible": False,
            "reason": f"Order is outside the {window_days}-day refund window.",
            "window_days": window_days,
            "days_since_delivery": days_since_delivery,
        }

# Quick sanity checks. The response should consider the region (US/EU/Global)
for oid in ["ORD-1001", "ORD-1002", "ORD-1003", "ORD-1004", "ORD-1005", "ORD-9999", "ORD-1006", "ORD-1007"]:
    o = lookup_order(oid)
    decision = check_refund_eligibility(o)
    print(oid, "->", decision)



In [ ]:
# Stage 3 - Step 2.3 - Refund Execution and Logging Function. Implement issue_refund() and a refund log. Sequence a refund is issued, the order is marked refunded, an audit log is created
# If you don’t want to “burn” ORD-1001 in tests, you can re-run the cell that defines orders to reset state
# Defines the issue_refund function, which simulates the successful refund process. This includes appending a record to the refund_log and updating the order's state by marking it as already refunded.

refund_log = []  # simple in-memory log

def issue_refund(order: dict, amount: float, today: date = None) -> dict:
    """
    Simulate issuing a refund:
      - append an entry to refund_log
      - mark order as already_refunded
    Returns the refund record.
    """
    if today is None:
        today = date.today()

    refund_record = {
        "order_id": order["order_id"],
        "customer_email": order["customer_email"],
        "amount": amount,
        "refunded_on": today,
    }
    refund_log.append(refund_record)
    order["already_refunded"] = True
    return refund_record

#Quick test on a copy to avoid messing up main data too early
test_order = lookup_order("ORD-1001")
print("Before:", test_order["already_refunded"])
test_refund = issue_refund(test_order, test_order["amount"])
print("Refund record:", test_refund)
print("After:", test_order["already_refunded"])
print("Refund log now:", refund_log)


In [ ]:
# Stage 3 - Step 2.4 - The Refund Decision Engine: The Deterministic Core
"""The refund_decision_engine is the single source of truth for all refund outcomes.
It implements the final business logic layer by sequencing three key, deterministic steps: data lookup, eligibility check, and conditional transaction execution (issuing the refund).
This function, operating purely on code, is immune to LLM hallucination, ensuring the core decision is always reliable, testable, and auditable.
This architecture clearly separates the LLM's language task from the code's action/decision task.
"""

def refund_decision_engine(order_id: str, claimed_defective: bool, today: date = None) -> dict:
    """
    Core refund decision logic:
      1. Look up order
      2. Override is_defective based on customer's claim (for demo)
      3. Check eligibility
      4. If eligible, issue refund
      5. Return structured decision
    """
    if today is None:
        today = date.today()

    order = lookup_order(order_id)

    if order is None:
        return {
            "status": "rejected",
            "reason": "Order not found.",
            "order_id": order_id,
            "refunded": False,
            "refund_amount": 0.0,
        }

    # For demo, treat customer's claim as the operative defect flag
    order["is_defective"] = claimed_defective

    eligibility = check_refund_eligibility(order, today)

    if not eligibility["eligible"]:
        return {
            "status": "rejected",
            "reason": eligibility["reason"],
            "order_id": order["order_id"],
            "refunded": False,
            "refund_amount": 0.0,
            "days_since_delivery": eligibility["days_since_delivery"],
            "window_days": eligibility["window_days"],
        }

    # Eligible → issue refund
    refund_record = issue_refund(order, order["amount"], today)

    return {
        "status": "approved",
            "reason": eligibility["reason"],
            "order_id": order["order_id"],
            "refunded": True,
            "refund_amount": refund_record["amount"],
            "days_since_delivery": eligibility["days_since_delivery"],
            "window_days": eligibility["window_days"],
        }

# Try some scenarios
tests = [
    ("ORD-1001", False),  # US, standard, likely inside 30 days
    ("ORD-1002", True),   # US, defective, likely outside 45 days
    ("ORD-1003", False),  # EU, inside 60 days
    ("ORD-1004", True),   # EU, outside 60 days
    ("ORD-1005", True),   # already refunded
    ("ORD-9999", False),  # unknown
    ("ORD-1006", False),  # recent, US
    ("ORD-1007", False),  # recent, EU
]

for oid, defect in tests:
    print("----")
    print("Request:", oid, "defective?", defect)
    print(refund_decision_engine(oid, defect))



In [ ]:
# Stage 3 - Step 3 - Takes the structured refund decision from the deterministic engine, the relevant refund policy retrieved via RAG and uses the LLM to explain the decision to the customer in natural language.
# This cell explains why the refund was approved or rejected, using: decision facts (status, days since delivery, window) | actual policy chunks (from FAISS + embeddings) | LLM phrasing (friendly explanation)
# This makes the agent - grounded in policy, transparent, customer-friendly, explainable, easier to debug, deterministic. It transforms  a structured refund decision into a human explanation that cites actual policy text retrieved via RAG.

def explain_refund_decision_with_policy(order_id: str, claimed_defective: bool, today: date = None) -> str:
    """
    Explain a refund decision to the customer, grounded in:
      - the structured decision from refund_decision_engine
      - the actual refund policy text retrieved via RAG
    """
    if today is None:
        today = date.today()

    # 1. Get structured decision
    decision = refund_decision_engine(order_id, claimed_defective, today)

    # 2. Build a short query to retrieve the relevant policy section
    order = lookup_order(order_id)
    if order is not None:
        country = order["country"]
        is_defective = claimed_defective

        if country == "EU":
            policy_query = "refund window for any product in Europe"
        else:
            if is_defective:
                policy_query = "refund window for defective products in the US and global rules"
            else:
                policy_query = "refund window for standard (non-defective) products in the US and global rules"
    else:
        policy_query = "summary of global refund policy rules and region-specific rules"

    context_docs = policy_retriever.invoke(policy_query)
    policy_context = "\n\n".join(d.page_content for d in context_docs)

    # 3. Build decision facts block
    facts = f"""
    Decision status: {decision['status']}
    Reason: {decision['reason']}
    Order ID: {decision['order_id']}
    Refunded: {decision['refunded']}
    Refund amount: {decision['refund_amount']}
    """

    if "days_since_delivery" in decision and decision["days_since_delivery"] is not None:
        facts += f"\nDays since delivery: {decision['days_since_delivery']}"
    if "window_days" in decision and decision["window_days"] is not None:
        facts += f"\nApplicable refund window (days): {decision['window_days']}"

    # 4. Prompt the LLM with both policy + decision facts
    prompt = f"""
You are a customer service agent explaining a refund decision to a customer.

You have two sources of information:

1) The official refund policy context (source of truth):
<refund_policy_context>
{policy_context}
</refund_policy_context>

2) The structured decision facts from the refund decision engine:
<decision_facts>
{facts}
</decision_facts>

Instructions:
- Explain the decision to the customer in clear, empathetic language.
- If the refund was approved, state that clearly and confirm the refund amount.
- If the refund was rejected, clearly explain why, using the policy rules where relevant
  (for example, mention the refund window, location and how many days have passed since delivery).
- Keep the explanation grounded in the refund policy context and the decision facts.
- Do NOT invent new policy rules beyond what is implied in the policy context.
- Do NOT change the underlying decision (status/approved vs rejected); you are only explaining it.
"""

    response = llm.invoke(prompt)
    return str(response)


In [ ]:
#Stage 3 - Step 4-  Smoke-tests the entire refund agent (decision logic + RAG + LLM) using real example orders to ensure everything works correctly.
# Deterministic logic makes the correct decision
# Policy RAG retrieves the right policy chunks
# LLM explains the decision clearly and accurately

print("=== US, standard, inside window ===")
print(explain_refund_decision_with_policy("ORD-1001", claimed_defective=False))
print("----")

print("=== US, defective, outside window ===")
print(explain_refund_decision_with_policy("ORD-1002", claimed_defective=True))
print("----")

print("=== EU, inside 60-day window ===")
print(explain_refund_decision_with_policy("ORD-1003", claimed_defective=False))
print("----")

print("=== EU, outside 60-day window ===")
print(explain_refund_decision_with_policy("ORD-1004", claimed_defective=True))
print("----")

print("=== Unknown order ===")
print(explain_refund_decision_with_policy("ORD-9999", claimed_defective=False))




In [ ]:
# Stage 3 - Step 5.1 - Mini evaluation report for the deterministic decision engine.
# This checks that refund_decision_engine returns the expected status for key scenarios.

test_cases = [
    {
        "name": "US standard, inside 30-day window",
        "order_id": "ORD-1001",
        "claimed_defective": False,
        "expected_status": "rejected",
    },
    {
        "name": "US defective, outside 45-day window",
        "order_id": "ORD-1002",
        "claimed_defective": True,
        "expected_status": "rejected",
    },
    {
        "name": "EU, inside 60-day window",
        "order_id": "ORD-1003",
        "claimed_defective": False,
        "expected_status": "rejected",
    },
    {
        "name": "EU, outside 60-day window",
        "order_id": "ORD-1004",
        "claimed_defective": True,
        "expected_status": "rejected",
    },
    {
        "name": "Already refunded order",
        "order_id": "ORD-1005",
        "claimed_defective": True,
        "expected_status": "rejected",
    },
    {
        "name": "Unknown order",
        "order_id": "ORD-9999",
        "claimed_defective": False,
        "expected_status": "rejected",
    },
    {
        "name": "US, outside 30 day",
        "order_id": "ORD-1006",
        "claimed_defective": False,
        "expected_status": "rejected",
    },
    {
        "name": "EU, inside the window",
        "order_id": "ORD-1007",
        "claimed_defective": False,
        "expected_status": "approved",
    },
]

def run_refund_eval():
    results = []
    print("=== Refund Decision Engine Evaluation ===")
    print(f"{'Case':40} {'Order':10} {'Expected':10} {'Actual':10} {'PASS/FAIL'}")
    print("-" * 80)

    for tc in test_cases:
        decision = refund_decision_engine(tc["order_id"], tc["claimed_defective"])
        actual_status = decision["status"]
        expected_status = tc["expected_status"]
        passed = (actual_status == expected_status)
        results.append(passed)

        print(
            f"{tc['name'][:38]:40} "
            f"{tc['order_id']:10} "
            f"{expected_status:10} "
            f"{actual_status:10} "
            f"{'PASS' if passed else 'FAIL'}"
        )

    print("-" * 80)
    total = len(results)
    passed_count = sum(results)
    print(f"Summary: {passed_count}/{total} passed")

run_refund_eval()


In [ ]:
# Stage 3 - Step 5.2 - Mini evaluation report for the LLM

import re

# Simple heuristics to detect approval vs rejection in the LLM explanation
def classify_explanation(text: str) -> str:
    """
    Very naive classifier:
    - If it contains clear acceptance phrases → 'approved'
    - If it contains clear denial phrases → 'rejected'
    - Otherwise → 'unknown'
    """
    lowered = text.lower()

    # Approval cues
    approve_patterns = [
        r"\byour refund (has been|is) approved\b",
        r"\bwe (will|are going to) process your refund\b",
        r"\byou will receive a refund\b",
    ]

    # Rejection cues
    reject_patterns = [
        r"\byour refund (request )?(has been|is) (denied|declined|rejected)\b",
        r"\bunfortunately, we (can't|cannot|are unable to) process your refund\b",
        r"\bnot able to issue a refund\b",
    ]

    for pat in approve_patterns:
        if re.search(pat, lowered):
            return "approved"

    for pat in reject_patterns:
        if re.search(pat, lowered):
            return "rejected"

    # Fallback: try a weaker heuristic
    if "unable to" in lowered or "cannot" in lowered and "refund" in lowered:
        return "rejected"

    return "unknown"


def run_llm_explanation_eval():
    print("=== LLM Explanation Evaluation ===")
    print(f"{'Case':40} {'Order':10} {'Expected':10} {'Predicted':10} {'PASS/FAIL'}")
    print("-" * 90)

    results = []
    for tc in test_cases:
        # Reuse the same test_cases from the deterministic eval (Stage 3 Step 8)
        explanation = explain_refund_decision_with_policy(
            tc["order_id"], tc["claimed_defective"]
        )
        predicted = classify_explanation(explanation)
        expected = tc["expected_status"]
        passed = (predicted == expected)

        results.append(passed)

        print(
            f"{tc['name'][:38]:40} "
            f"{tc['order_id']:10} "
            f"{expected:10} "
            f"{predicted:10} "
            f"{'PASS' if passed else 'FAIL'}"
        )

    print("-" * 90)
    total = len(results)
    passed_count = sum(results)
    print(f"Summary: {passed_count}/{total} passed (LLM explanation alignment)")

run_llm_explanation_eval()


In [ ]:
# Stage 4 - Step 1 - Local Timing instrumentation. Simple timing wrapper for the full agent - Add Timing & Latency Observability in Colab
# Local instrumentation - Wrap the agent call in a timing function to break down deterministic vs RAG+LLM latency. We'll add Langsmith in later sections.
# Instrumentation logic when we need to run the agent with the stopwatch on for profiling. The core logic stays separate/clean and focused on “what” the system does
# Can experiment with different profiling strategies without touching the core logic and for most runs we don't need timing overhead/noise

import time

def timed_explain_refund(order_id: str, claimed_defective: bool):
    """
    Measure latency for:
      - deterministic decision engine
      - RAG + LLM explanation
    Returns both the result and a timing breakdown.
    """
    t0 = time.time()

    # 1) Deterministic decision (rules)
    t_decision_start = time.time()
    decision = refund_decision_engine(order_id, claimed_defective)
    t_decision_end = time.time()

    # 2) RAG + LLM explanation
    t_explain_start = time.time()
    explanation = explain_refund_decision_with_policy(order_id, claimed_defective)
    t_explain_end = time.time()

    total = t_explain_end - t0
    decision_time = t_decision_end - t_decision_start
    explain_time = t_explain_end - t_explain_start

    return {
        "order_id": order_id,
        "claimed_defective": claimed_defective,
        "decision": decision,
        "explanation": explanation,
        "timing": {
            "total_sec": total,
            "decision_sec": decision_time,
            "rag_plus_llm_sec": explain_time,
        },
    }

# Quick example
res = timed_explain_refund("ORD-1001", claimed_defective=False)
print("Decision:", res["decision"])
print("\nExplanation:\n", res["explanation"])
print("\nTiming (seconds):", res["timing"])


In [ ]:
# Stage 4 - Step 2 - Let's check latency across various scenarios using the above local instrumentation

def run_latency_suite(cases):
    print("=== Latency evaluation for refund agent ===")
    rows = []
    for tc in cases:
        result = timed_explain_refund(tc["order_id"], tc["claimed_defective"])
        timing = result["timing"]
        rows.append({
            "name": tc["name"],
            "order_id": tc["order_id"],
            "total_sec": timing["total_sec"],
            "decision_sec": timing["decision_sec"],
            "rag_plus_llm_sec": timing["rag_plus_llm_sec"],
        })

    # Pretty-print as a small table
    print(f"{'Case':40} {'Order':10} {'Total(s)':10} {'Decision(s)':12} {'RAG+LLM(s)':12}")
    print("-" * 90)
    for r in rows:
        print(
            f"{r['name'][:38]:40} "
            f"{r['order_id']:10} "
            f"{r['total_sec']:<10.3f} "
            f"{r['decision_sec']:<12.3f} "
            f"{r['rag_plus_llm_sec']:<12.3f}"
        )

# Reuse the existing test_cases from Stage 3 evals
run_latency_suite(test_cases)




In [ ]:
# Stage 4 - Step 3 - Wiring up Langsmith for tracing and prove Colab ↔ LangSmith ↔ Project all work. Uncomment only if for any reason runs are not being recorded in Langsmith

import os
from datetime import datetime
from langsmith import Client


# Create LangSmith client (uses env vars by default)
client = Client()

project_name = os.getenv("LANGCHAIN_PROJECT") or "refund-agent-demo"
print(f"Ensuring project exists: {project_name!r}")

# Newer client uses has_project / create_project / read_project
if client.has_project(project_name=project_name):
    project = client.read_project(project_name=project_name)
    print("Project already exists.")
else:
    try:
        project = client.create_project(
            project_name=project_name,
            description="Refund agent demo project from Google Colab.",
        )
        print("Project created.")
    except Exception as e:
        print(f"Error creating project: {e}")
        project = None # Ensure project is None if creation fails


if project:
    print("  Project ID:  ", project.id)
    print("  Project Name:", project.name)
else:
    print("  Could not retrieve or create project.")
print()

# Create a manual test run so you can see it in the LangSmith UI
# print("Creating a manual test run in LangSmith...")

# Add a try-except block and check if 'run' is None

try:
    run = client.create_run(
        name="manual-refund-test",
        run_type="chain",
        project_name=project_name,
        inputs={"order_id": "ORD-1001", "claimed_defective": False},
        end_time=datetime.utcnow(),
    )

    if run:
        print("\n✅ Now go to LangSmith → Tracing Projects →"
              f" '{project_name}' → Runs and look for 'manual-refund-test'.")
    else:
        print("Manual run could not be created. `client.create_run()` returned None.")
except Exception as e:
    print(f"Error creating manual run: {e}")


In [ ]:
#Stage 4 - Step 4 = Enabling End-to-End Observability with LangSmith
""" This step wraps the entire agent workflow, run_refund_agent, with the @traceable decorator from LangSmith.
This ensures that every sub-step—from the RAG retrieval and LLM call to the final decision execution—is captured and logged.
This end-to-end tracing is essential for debugging, monitoring performance, evaluating the quality of the generated answers, and gaining visibility into the agent's complex decision-making process.
"""

from langsmith import traceable

@traceable(
    name="refund-agent-end-to-end",
    project_name=os.getenv("LANGCHAIN_PROJECT", "refund-agent-demo"),
    run_type="chain",
)
def run_refund_agent(order_id: str, claimed_defective: bool) -> str:
    return explain_refund_decision_with_policy(order_id, claimed_defective)

print("Calling run_refund_agent with LangSmith tracing enabled...")

result = run_refund_agent("ORD-1001", False)
print(result[:400], "...")


In [ ]:
#Stage 5 - Step 1 - Preparing the Managed Vector Database Option (Pinecone). This is an alternative managed vectorDB - Pinecone. We started off with in-memory FAISS, however, also making this option available
"""This section installs the required libraries for connecting to the Pinecone managed vector database.
By installing pinecone-client and langchain-pinecone, we establish the foundation for the alternative, cloud-hosted vector store.
This allows the architecture to seamlessly switch from the local, in-memory FAISS index to a scalable, production-grade Pinecone index via environment variables, as planned.
"""

# Install Pinecone & LangChain integration

!pip install -qU pinecone-client pinecone-client[grpc] langchain-pinecone
print("Pinecone + langchain-pinecone installed.")

# Sanity-check that Pinecone + LangChain integration installed correctly

try:
    from pinecone import Pinecone
    from langchain_pinecone import PineconeVectorStore
    print("✅ Pinecone and langchain-pinecone imported successfully.")
except Exception as e:
    print("❌ Import failed:", e)



In [ ]:
# Stage 5 - Step 2 - Configure Pinecone credentials & index name. Make sure to create your free Pinecone account and create an index prior to this.
# I selected NVIDIA hosted - llama-text-embed-v2 and AWS
# This centralizes the Pinecone config so later functions can read from os.environ and PINECONE_INDEX_NAME.

import os

# Set this once per runtime. Load it from Google colab secrets.
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

PINECONE_INDEX_NAME = "refund-policy-index"  # must exist in Pinecone console

print("Pinecone config set. Index name:", PINECONE_INDEX_NAME)


In [ ]:
#Stage 5 – Step 3: Build a Pinecone vector store from the policy docs
"""
Building the Persistent, Scalable Vector Store:
We define a utility to establish the connection with Pinecone and create the persistent vector index.
Unlike the in-memory FAISS, this step ensures that the policy embeddings are stored externally, providing scalability and data persistence for large-scale or production RAG deployments.
The resulting pinecone_retriever is configured for use in the agent's swappable architecture.
"""

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os

def build_pinecone_vectorstore(docs, embeddings, index_name: str):
    """
    Create or reuse a Pinecone index and safely load policy docs into it
    by clearing the namespace first (for re-indexing purposes).
    """
    api_key = os.getenv("PINECONE_API_KEY")
    if not api_key:
        raise ValueError("PINECONE_API_KEY is not set in environment variables.")

    # --- PINE CONE SETUP ---
    pc = Pinecone(api_key=api_key)

    # 1. Define Target
    NAMESPACE = "refund-policy"
    index = pc.Index(index_name)

    print(f"Using Pinecone index: {index_name}, Namespace: {NAMESPACE}")

    # --- RE-INDEXING STEP: DELETE OLD VECTORS ---
    print(f"Deleting all vectors in namespace '{NAMESPACE}' for clean re-indexing...")
    try:
        # Use the raw Pinecone client to delete all existing vectors in the namespace
        index.delete(delete_all=True, namespace=NAMESPACE)
        print("Deletion successful.")
    except Exception as e:
        print(f"Warning: Could not delete existing vectors. Error: {e}")
        # Continue execution, but a warning is needed.

    # --- UPLOAD NEW CHUNKS (UPSERT) ---
    print(f"Uploading {len(docs)} new chunks with improved segmentation...")

    # Build a LangChain vector store from documents (this performs the upsert)
    vectorstore = PineconeVectorStore.from_documents(
        docs,
        embeddings,
        index_name=index_name,
        namespace=NAMESPACE,
    )

    # 2. Add an optional count check (good practice)
    vector_count = index.describe_index_stats()['namespaces'][NAMESPACE]['vector_count']
    print(f"Pinecone index now contains {vector_count} vectors.")

    return vectorstore

# Build Pinecone-based vector store & retriever
# Rerunning this line now performs a safe delete-and-upsert
pinecone_vectorstore = build_pinecone_vectorstore(policy_docs, embeddings, "refund-policy-index")

# Set the Pinecone retriever to use the same k=5 as FAISS
pinecone_retriever = pinecone_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},  # Explicitly setting k=5
)
print("Pinecone vector store and retriever initialized.")

In [ ]:
# Stage 5 - Step 4 - Switch between FAISS and Pinecone retrievers
"""
This critical step realizes the flexible vector store architecture by creating the active_retriever variable.
The value of the boolean flag USE_PINECONE acts as the environment control, allowing the RAG pipeline to dynamically switch between the local, in-memory FAISS retriever (for fast development/testing) and the cloud-hosted, scalable Pinecone retriever (for production/large-scale use).
This centralizes the choice of the vector database backend for the entire agent.
"""

faiss_retriever = policy_retriever  # existing FAISS-based retriever

USE_PINECONE = True  # set to False to use FAISS instead

active_retriever = pinecone_retriever if USE_PINECONE else faiss_retriever

print("Active retriever backend:", "Pinecone" if USE_PINECONE else "FAISS")


In [ ]:
# Stage 5 - Step 5 - Quick compare: FAISS vs Pinecone retrieval answers / Comparative Validation of Vector Store Backends
"""
A series of test questions are run against both the FAISS and Pinecone retrievers to gain confidence in the system's ability to seamlessly switch between the two.
Successful results (i.e., identical LLM answers from both backends) confirm that the indexing and retrieval layer is independent of the vector store technology, paving the way for a smooth transition from development (FAISS) to production (Pinecone).
"""

test_questions = [
    "What is the refund window for standard products in the US?",
    "How long do customers in Europe have to request a refund?",
    "What happens if the product is defective?",
]

def run_backend_test(use_pinecone: bool):
    global active_retriever
    active_retriever = pinecone_retriever if use_pinecone else faiss_retriever
    backend_name = "PINECONE" if use_pinecone else "FAISS"
    print(f"\n=== Testing backend: {backend_name} ===")

    for q in test_questions:
        print(f"\nQ: {q}")
        ans = refund_policy_rag_answer(q)
        print(f"A: {ans[:400]}...")

# Test FAISS
run_backend_test(use_pinecone=False)

# Test Pinecone
run_backend_test(use_pinecone=True)


In [ ]:
# Stage 5.5 - LangSmith-instrumented RAG comparison: FAISS vs Pinecone / Performance and Tradeoff Analysis with LangSmith Tracing
"""
Instrumented the RAG policy layer to compare FAISS vs Pinecone both locally and in LangSmith.
Used a swappable retriever abstraction and then wrapped FAISS and Pinecone runs in separate traced functions so we can see per-backend latency,
LLM calls, and outputs in the LangSmith UI. This let us reason about the tradeoffs between an in-memory vector store and a managed vector DB
without changing the core agent logic.
This step finalizes the validation of the swappable vector store architecture by performing an A/B performance test between the local FAISS index and the managed Pinecone service.
Two dedicated @traceable functions are created to force the use of each backend, ensuring all latency and execution details (including retriever time, LLM calls, and final output) are logged separately in LangSmith.
This allows for a robust, quantitative comparison of the in-memory (FAISS) vs. cloud-managed (Pinecone) performance tradeoffs without altering the core RAG logic.
"""

import os
import time
from langsmith import traceable

TEST_QUESTIONS = [
    "What is the refund window for standard products in the US?",
    "How long do customers in Europe have to request a refund?",
    "What happens if the product is defective?",
]


@traceable(
    name="refund-policy-rag-faiss",
    project_name=os.getenv("LANGCHAIN_PROJECT", "refund-agent-demo"),
    run_type="chain",
)
def traced_refund_policy_rag_faiss(question: str) -> str:
    """Run the RAG policy answer using FAISS as the backend."""
    global active_retriever
    active_retriever = faiss_retriever  # force FAISS

    start = time.perf_counter()
    answer = refund_policy_rag_answer(question)
    elapsed = time.perf_counter() - start
    print(f"[FAISS] RAG latency: {elapsed*1000:.1f} ms")

    return answer


@traceable(
    name="refund-policy-rag-pinecone",
    project_name=os.getenv("LANGCHAIN_PROJECT", "refund-agent-demo"),
    run_type="chain",
)
def traced_refund_policy_rag_pinecone(question: str) -> str:
    """Run the RAG policy answer using Pinecone as the backend."""
    global active_retriever
    active_retriever = pinecone_retriever  # force Pinecone

    start = time.perf_counter()
    answer = refund_policy_rag_answer(question)
    elapsed = time.perf_counter() - start
    print(f"[PINECONE] RAG latency: {elapsed*1000:.1f} ms")

    return answer


print("Running FAISS vs Pinecone RAG comparison with LangSmith traces...\n")

for q in TEST_QUESTIONS:
    print("\n==============================")
    print("Question:", q)

    print("\n--- FAISS backend ---")
    ans_faiss = traced_refund_policy_rag_faiss(q)
    print("Answer (FAISS):", ans_faiss[:300], "...")

    print("\n--- PINECONE backend ---")
    ans_pinecone = traced_refund_policy_rag_pinecone(q)
    print("Answer (PINECONE):", ans_pinecone[:300], "...")


In [ ]:
# I started with a notebook-only agent with in-memory FAISS & RAG, Core business logic, LLM explanation/response to user
# Then, made it observable with local and Langsmith instrumentation, then made its vector backend pluggable with cloud-hosted Pinecone,
# Stage 6 - Step 1 - In stage 1 below, we define & test the agent API interface locally first before we expose it externally.
# Stage 2 will be the actual exposure via MCP server and will be in a different .py file directly in github repository

from typing import Dict, Any

def refund_agent_api(order_id: str, claimed_defective: bool) -> Dict[str, Any]:
    """
    Clean API wrapper for MCP.

    Runs:
      - deterministic refund_decision_engine
      - policy-grounded LLM explanation

    Returns:
      A JSON-serializable dict suitable for MCP tool responses.
    """
    # 1. Deterministic policy logic
    decision = refund_decision_engine(order_id, claimed_defective)

    # 2. Natural-language explanation (RAG + LLM)
    explanation = explain_refund_decision_with_policy(order_id, claimed_defective)

    # Normalize decision fields for MCP clients
    result = {
        "order_id": order_id,
        "claimed_defective": claimed_defective,
        "status": decision.get("status"),                 # "approved" or "rejected"
        "reason": decision.get("reason"),
        "refunded": decision.get("refunded"),             # boolean
        "refund_amount": decision.get("refund_amount"),
        "days_since_delivery": decision.get("days_since_delivery"),
        "window_days": decision.get("window_days"),
        "raw_decision": decision,                         # deterministic structured output
        "explanation": explanation,                       # LLM output
    }

    return result

# Quick sanity check
print("=== refund_agent_api sanity test ===")
print(refund_agent_api("ORD-1001", claimed_defective=False))

